# Phase 05 — Fine-tune `bge-small-en-v1.5` on hard-negative triplets

**Runtime: T4 GPU.** Set via `Runtime → Change runtime type → T4 GPU`.

## What this notebook does

1. Clones the V3 repo (which has `artifacts/hard_negatives_v3.jsonl` committed).
2. Installs the minimal deps not already on Colab.
3. Runs `src/retrieval/finetune/train_bge.py` with `--fp16` on the T4.
4. Optionally re-encodes the corpus with the fine-tuned model (requires uploading `artifacts/corpus.parquet`).
5. Zips the checkpoint + (optional) fine-tune embeddings/FAISS index and downloads to your machine.

Expected wall time on T4: **~10–15 min training** + ~3 min optional re-encoding.

## What happens locally afterwards

1. Unzip the downloaded artifacts into the v3 repo (`checkpoints/bge-small-v3-hn/` and `artifacts/corpus_ft.faiss` etc.).
2. Run `python -m src.eval.retrieval_eval_finetune --n 200` to compare base vs fine-tune on HoVer dev.
3. The decision (ship base or fine-tune) lands in `docs/PHASE_05_DECISION.md` and `configs/default.yaml`.

## 1. Verify GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (training will be slow)"}')

## 2. Clone repo + install deps

If you've cloned this repo into Colab before, the `git clone` cell will fail harmlessly and the rest still works.

In [ ]:
!git clone https://github.com/Sar-Ahmed/Adversarial-Multi-Hop-Fact-Verification.git
%cd Adversarial-Multi-Hop-Fact-Verification
!git pull

In [ ]:
# Colab already has torch + transformers; we add sentence-transformers and a few light deps.
!pip install -q sentence-transformers==3.4.1 loguru==0.7.2 typer==0.12.5 pydantic==2.9.2 pyyaml==6.0.2 datasets==2.20.0 faiss-cpu==1.10.0

## 3. Verify triplets are committed and load-ready

In [ ]:
import os, json
TRIPS = 'artifacts/hard_negatives_v3.jsonl'
assert os.path.exists(TRIPS), f'{TRIPS} missing — pull the latest commit from master'
n_lines = sum(1 for _ in open(TRIPS, encoding='utf-8'))
print(f'{TRIPS}: {os.path.getsize(TRIPS)/1e6:.1f} MB, {n_lines:,} triplets')
with open(TRIPS, encoding='utf-8') as f:
    sample = json.loads(f.readline())
print('sample:', json.dumps(sample, indent=2)[:300])

## 4. Train

Hyperparameters mirror `src/retrieval/finetune/train_bge.py` but `batch_size=64` for T4 (vs CPU default 32) and `--fp16` enabled.

T4 wall time: ~10-15 min for 30-40k triplets at batch 64.

In [ ]:
!python -m src.retrieval.finetune.train_bge --epochs 1 --batch-size 64 --learning-rate 2e-5 --fp16

In [ ]:
# Sanity check: load the saved checkpoint and encode a tiny query.
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('checkpoints/bge-small-v3-hn', device='cuda')
v = m.encode(['Christopher Nolan directed Inception.'], normalize_embeddings=True)
print('vector shape:', v.shape, 'norm:', float((v**2).sum()**0.5))

## 5. (Optional) Re-encode the corpus on the T4

`artifacts/corpus.parquet` (12 MB) is gitignored. Upload it via the cell below to re-encode here, or skip and run `python -m src.data.encode_corpus --model checkpoints/bge-small-v3-hn --suffix _ft` on your local machine afterwards (~30-45 min on CPU).

T4 re-encoding wall time: ~3 min.

In [ ]:
# Skip this cell if you'd rather re-encode locally.
from google.colab import files
print('Select your local artifacts/corpus.parquet (12 MB):')
uploaded = files.upload()
import shutil, os
for name in uploaded:
    shutil.copy(name, 'artifacts/corpus.parquet')
    os.remove(name)
print('uploaded:', os.path.getsize('artifacts/corpus.parquet')/1e6, 'MB')

In [ ]:
# Re-encode using the fine-tuned model. Suffix '_ft' keeps the Phase 01 baseline files untouched.
!python -m src.data.encode_corpus --model checkpoints/bge-small-v3-hn --suffix _ft --device cuda

## 6. Zip and download

Pulls down a single zip with everything Phase 05 needs locally. Unzip into the repo root.

In [ ]:
import os
to_zip = ['checkpoints/bge-small-v3-hn', 'artifacts/retriever_finetune_log.jsonl']
for ft_artifact in ['artifacts/corpus_embeddings_ft.npy', 'artifacts/corpus_ft.faiss', 'artifacts/corpus_encoding_meta_ft.json']:
    if os.path.exists(ft_artifact):
        to_zip.append(ft_artifact)
print('zipping:', to_zip)
import subprocess
subprocess.check_call(['zip', '-r', 'phase05_artifacts.zip'] + to_zip)
print('zip size:', os.path.getsize('phase05_artifacts.zip')/1e6, 'MB')

In [ ]:
from google.colab import files
files.download('phase05_artifacts.zip')